![Universidad Iberoamericana del Ecuador](https://raw.githubusercontent.com/andresrubiop/ml-unibe-estudiantes/main/utils/logo-unibe-color.png)

<div style="background:#004890;color:#FFFFFF;padding:14px 18px;border-radius:10px;margin:6px 0 12px 0;border-bottom:4px solid #FCC000"><div style="font-size:12px;letter-spacing:.08em;text-transform:uppercase;opacity:.9">Machine Learning · Sesión 2 · Actividad calificada · aprendizaje autónomo</div><div style="font-size:22px;font-weight:700;margin-top:4px">Deber 3 · Ruta B · Un flujo que escala con PyTorch</div><div style="font-size:12px;opacity:.9;margin-top:4px">Posgrado · Maestría en Big Data y Ciencia de Datos · UNIB.E</div></div>

| | |
|---|---|
| **Qué entregas** | **Un solo notebook ejecutado** (`.ipynb`): este (ruta B, PyTorch) **o** el de la ruta A (Spark). Renómbralo `D3_Apellido_Nombre.ipynb` y súbelo al aula virtual. No hay que subir nada más. |
| **Fecha** | **Jueves 19 de noviembre de 2026** |
| **Tiempo estimado** | 3 a 4 horas (el entrenamiento tarda unos 5 minutos en CPU y menos de 2 con GPU) |
| **Peso** | 1/3 de los 2 puntos de aprendizaje autónomo · rúbrica al final de este notebook y en el aula |
| **Uso de IA** | **Permitido sin límite**: Claude Code, Codex, ChatGPT, Gemini en Colab, Copilot u otra, siempre que lo **declares**. Este deber incluye un **desafío con IA** obligatorio (parte 6). |
| **Teoría** | Manual M2 §10–§11 · laboratorio L2.4 y tutoría de deep learning (LT.1) |
| **Datos** | **Fashion-MNIST completo**: 60 000 fotos de entrenamiento y 10 000 de prueba. La carga ya está hecha. |

**Qué vas a hacer.** Entrenar con **PyTorch** dos redes (un perceptrón multicapa y una red convolucional) sobre los datos completos, con validación y parada temprana, guardar el mejor modelo y documentarlo. Es el flujo que se lleva a una GPU o a un clúster.

**Partes:** 0 · cómo trabajar → 1 · configuración → 2 · datos y `DataLoader` (hecho) → 3 · dos arquitecturas → 4 · entrenamiento con validación y parada temprana → 5 · curvas, prueba final y modelo guardado → 6 · desafío con IA (script y SLURM) → 7 · reflexión → declaración y lista de control.

## 0 · Cómo trabajar este notebook

1. **Guarda tu copia.** En Colab: *Archivo → Guardar una copia en Drive*.
2. **(Opcional) Activa una GPU:** *Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU T4*. Sin GPU también funciona, solo más lento.
3. **Ejecuta de arriba hacia abajo** y completa las celdas `# TODO`. Las celdas **✅ Autoverificación** te dicen si vas bien.
4. **Responde en las celdas ✍️** con tus palabras.
5. **Antes de entregar:** *Reiniciar y ejecutar todas*; luego *Archivo → Descargar → Descargar .ipynb*, renómbralo `D3_Apellido_Nombre.ipynb` y súbelo.

## 1 · Configuración

In [ ]:
import copy
import gzip
import os
import time
import urllib.request

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, Subset, TensorDataset, random_split

SEED = 2026
torch.manual_seed(SEED)
np.random.seed(SEED)
DISPOSITIVO = "cuda" if torch.cuda.is_available() else "cpu"
CACHE = os.path.expanduser("~/.cache/ml-unibe")
URL = "https://storage.googleapis.com/tensorflow/tf-keras-datasets/"
print(f"PyTorch {torch.__version__} · dispositivo: {DISPOSITIVO} · semilla {SEED}")

## 2 · Datos y `DataLoader` (ya resuelto)

Cargamos las 70 000 fotos como tensores con forma `(n, 1, 28, 28)` (un canal de gris). Del entrenamiento separamos **5 000 fotos de validación fijas** (con semilla) para decidir cuándo parar; la prueba no se toca hasta la parte 5.

Un `DataLoader` entrega los datos en **lotes** (aquí de 128) y **baraja** el entrenamiento en cada época: así PyTorch puede trabajar con datos que no caben enteros en la GPU.

<details><summary><b>Ayuda: otras formas de cargar datos en PyTorch</b></summary>

- **Con torchvision:** `torchvision.datasets.FashionMNIST(root="datos", train=True, download=True, transform=torchvision.transforms.ToTensor())`.
- **Tus propias imágenes en carpetas** (una carpeta por clase): `torchvision.datasets.ImageFolder("ruta", transform=...)`.
- **Una tabla (CSV o Parquet):** léela con pandas y conviértela con `TensorDataset(torch.tensor(X, dtype=torch.float32), torch.tensor(y))`.
</details>

In [ ]:
def descargar(nombre):
    os.makedirs(CACHE, exist_ok=True)
    ruta = os.path.join(CACHE, nombre)
    if not os.path.exists(ruta):
        urllib.request.urlretrieve(URL + nombre, ruta)
    return ruta


def cargar(nombre_img, nombre_etq):
    with gzip.open(descargar(nombre_img)) as f:
        X = np.frombuffer(f.read(), np.uint8, offset=16).reshape(-1, 1, 28, 28)
    with gzip.open(descargar(nombre_etq)) as f:
        y = np.frombuffer(f.read(), np.uint8, offset=8)
    return TensorDataset(torch.tensor(X, dtype=torch.float32) / 255.0, torch.tensor(y, dtype=torch.long))


PRENDAS = ["camiseta", "pantalón", "suéter", "vestido", "abrigo", "sandalia", "camisa", "zapatilla", "bolso", "botín"]
entrenamiento_completo = cargar("train-images-idx3-ubyte.gz", "train-labels-idx1-ubyte.gz")
prueba = cargar("t10k-images-idx3-ubyte.gz", "t10k-labels-idx1-ubyte.gz")
entrenamiento, validacion = random_split(entrenamiento_completo, [55_000, 5_000], generator=torch.Generator().manual_seed(SEED))

carga_tr = DataLoader(entrenamiento, batch_size=128, shuffle=True, generator=torch.Generator().manual_seed(SEED))
carga_va = DataLoader(validacion, batch_size=1000)
carga_tr_eval = DataLoader(Subset(entrenamiento, range(5_000)), batch_size=1000)
carga_te = DataLoader(prueba, batch_size=1000)
x, y = next(iter(carga_tr))
print("entrenamiento:", len(entrenamiento), "· validación:", len(validacion), "· prueba:", len(prueba))
print("un lote:", tuple(x.shape), "imágenes y", tuple(y.shape), "etiquetas")

## 3 · Dos arquitecturas

**Tu tarea:** define dos redes que reciban `(lote, 1, 28, 28)` y devuelvan 10 puntajes (uno por prenda):

1. **Perceptrón multicapa (MLP):** `nn.Flatten()` → `nn.Linear(784, 256)` → `nn.ReLU()` → `nn.Dropout(0.2)` → `nn.Linear(256, 10)`. Ya está hecho como ejemplo.
2. **Red convolucional (CNN):** dos bloques `nn.Conv2d` → `nn.ReLU()` → `nn.MaxPool2d(2)` (por ejemplo, 32 y 64 filtros de 3 × 3 con `padding=1`), y luego `nn.Flatten()` → `nn.Linear(64 * 7 * 7, 128)` → `nn.ReLU()` → `nn.Linear(128, 10)`.

**Pista:** cada `MaxPool2d(2)` divide el tamaño por dos: 28 → 14 → 7. Por eso la capa lineal recibe `64 * 7 * 7` valores.

In [ ]:
def crear_mlp():
    return nn.Sequential(nn.Flatten(), nn.Linear(784, 256), nn.ReLU(), nn.Dropout(0.2), nn.Linear(256, 10))


def crear_cnn():
    # TODO: completa la red convolucional.
    return nn.Sequential(
        nn.Conv2d(1, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        ...,                                 # segundo bloque convolucional
        nn.Flatten(), ...                    # capas lineales hasta 10 salidas
    )


for nombre, crear in (("MLP", crear_mlp), ("CNN", crear_cnn)):
    red = crear()
    print(f"{nombre}: salida {tuple(red(x[:4]).shape)} · {sum(p.numel() for p in red.parameters()):,} parámetros")

**✅ Autoverificación de la parte 3**

In [ ]:
assert crear_cnn()(x[:4]).shape == (4, 10), "La CNN debe devolver 10 puntajes por imagen."
assert any(isinstance(m, nn.Conv2d) for m in crear_cnn().modules()), "La CNN necesita capas Conv2d."
print("✓ Parte 3 correcta")

## 4 · Entrenar con validación y parada temprana

La función `evaluar` ya está hecha. **Tu tarea** es completar el ciclo de entrenamiento de `entrenar`:

- en cada lote: poner los gradientes en cero, calcular la pérdida, propagar hacia atrás y dar un paso del optimizador;
- al final de cada época: medir la exactitud de validación y **guardar una copia del mejor modelo**;
- **parada temprana:** si la validación no mejora en `paciencia` épocas seguidas, parar.

In [ ]:
def evaluar(red, carga):
    """Exactitud y pérdida media de la red en un DataLoader."""
    red.eval()
    aciertos, perdida, n = 0, 0.0, 0
    with torch.no_grad():
        for xb, yb in carga:
            xb, yb = xb.to(DISPOSITIVO), yb.to(DISPOSITIVO)
            salida = red(xb)
            perdida += nn.functional.cross_entropy(salida, yb, reduction="sum").item()
            aciertos += (salida.argmax(1) == yb).sum().item()
            n += len(yb)
    return aciertos / n, perdida / n


def entrenar(red, epocas=12, paciencia=3, lr=1e-3):
    red = red.to(DISPOSITIVO)
    opt = torch.optim.Adam(red.parameters(), lr=lr)
    historia, mejor_va, mejor_estado, sin_mejora = [], 0.0, None, 0
    for epoca in range(1, epocas + 1):
        red.train()
        t0 = time.time()
        for xb, yb in carga_tr:
            xb, yb = xb.to(DISPOSITIVO), yb.to(DISPOSITIVO)
            # TODO: un paso de entrenamiento (cero gradientes, pérdida, backward, step).
            opt.zero_grad()
            perdida = ...                    # nn.functional.cross_entropy(red(xb), yb)
            ...                              # propaga hacia atrás
            ...                              # paso del optimizador
        acc_tr, loss_tr = evaluar(red, carga_tr_eval)       # 5 000 fotos de entrenamiento, para comparar con la validación
        acc_va, loss_va = evaluar(red, carga_va)
        historia.append({"época": epoca, "exactitud entrenamiento": acc_tr, "exactitud validación": acc_va,
                         "pérdida entrenamiento": loss_tr, "pérdida validación": loss_va})
        print(f"época {epoca:2d} · entrenamiento {acc_tr:.3f} · validación {acc_va:.3f} · {time.time() - t0:.0f} s")
        # TODO: guarda el mejor modelo y aplica la parada temprana.
        if acc_va > mejor_va:
            mejor_va, mejor_estado, sin_mejora = acc_va, copy.deepcopy(red.state_dict()), 0
        else:
            ...                              # suma una época sin mejora y, si llega a `paciencia`, sale del ciclo
    red.load_state_dict(mejor_estado)
    return red, pd.DataFrame(historia)


torch.manual_seed(SEED)
mlp, hist_mlp = entrenar(crear_mlp())
torch.manual_seed(SEED)
cnn, hist_cnn = entrenar(crear_cnn())

**✅ Autoverificación de la parte 4**

In [ ]:
assert hist_mlp["exactitud validación"].max() > 0.85, "El MLP debería superar el 85 % en validación."
assert hist_cnn["exactitud validación"].max() > 0.88, "La CNN debería superar el 88 % en validación."
print("✓ Parte 4 correcta")

## 5 · Curvas, prueba final y modelo guardado

**Tu tarea:**
1. Mira las **curvas de aprendizaje** (ya dibujadas) y responde la pregunta sobre sobreajuste.
2. Evalúa **una sola vez** las dos redes en la prueba.
3. **Guarda** los pesos de la mejor red con `torch.save(red.state_dict(), "fashion_cnn.pt")`, **recárgalos** en una red nueva y comprueba que da la misma exactitud.

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(10, 3.2))
for ax, (nombre, h) in zip(axs, (("MLP", hist_mlp), ("CNN", hist_cnn))):
    ax.plot(h["época"], h["exactitud entrenamiento"], "o-", label="entrenamiento")
    ax.plot(h["época"], h["exactitud validación"], "o-", label="validación")
    ax.set_title(nombre); ax.set_xlabel("época"); ax.legend()
plt.show()

In [ ]:
# TODO: exactitud en la prueba de las dos redes; guarda, recarga y comprueba la mejor.
acc_te_mlp, _ = evaluar(mlp, carga_te)
acc_te_cnn, _ = ...
torch.save(..., "fashion_cnn.pt")
recargada = crear_cnn().to(DISPOSITIVO)
recargada.load_state_dict(torch.load("fashion_cnn.pt", map_location=DISPOSITIVO))
acc_recargada, _ = ...
print(f"prueba · MLP {acc_te_mlp:.3f} · CNN {acc_te_cnn:.3f} · CNN recargada {acc_recargada:.3f}")

**✅ Autoverificación de la parte 5**

In [ ]:
assert os.path.exists("fashion_cnn.pt"), "Guarda los pesos con torch.save."
assert abs(acc_recargada - acc_te_cnn) < 1e-9, "La red recargada debe dar exactamente la misma exactitud."
print("✓ Parte 5 correcta")

✍️ **Pregunta 5.** ¿Alguna red sobreajusta (entrenamiento sube y validación se estanca)? ¿En qué época paró cada una? ¿Cuánto gana la CNN sobre el MLP y a qué costo (parámetros, tiempo)?

*Tu respuesta aquí.*

✍️ **Ficha del modelo** (completa con tus cifras):

| | |
|---|---|
| **Modelo** | *CNN de dos bloques convolucionales, N parámetros* |
| **Datos** | *Fashion-MNIST: 55 000 entrenamiento, 5 000 validación, 10 000 prueba* |
| **Desempeño** | *exactitud en la prueba; prendas que más confunde* |
| **Uso previsto** | *clasificar fotos de prendas de 28 × 28 en escala de grises* |
| **Limitaciones** | *qué no debería hacerse con este modelo* |

## 6 · Desafío con IA: script reproducible y guion de SLURM

El desafío obligatorio de este deber. Pide a un asistente de IA (Claude Code, Codex, ChatGPT, Gemini…) que convierta tu entrenamiento en:

1. un **script de Python** (`entrenar_cnn.py`) que reciba por argumentos las épocas, el tamaño de lote y la ruta de salida, fije las semillas y guarde los pesos y la exactitud;
2. un **guion de SLURM** (`entrenar.slurm`) que pida **una GPU**, con `[COMPLETAR]` en lo que dependa del centro de cómputo (partición, cuenta, módulos).

**Verifica** lo que te entrega: ¿el script corre aquí con `!python entrenar_cnn.py --epocas 1`? ¿Fija las semillas? ¿El guion pide la GPU correctamente? Pega abajo la versión final (ya corregida por ti).

In [ ]:
# TODO: pega aquí el script final (puedes escribirlo a un archivo con %%writefile en otra celda) y el guion de SLURM.
SLURM = """..."""
print(SLURM)

## 7 · Reflexión (≈ 150 palabras)

✍️ ¿Qué cambiaría si los datos fueran **100 veces más** (6 millones de fotos)? Piensa en memoria, lectura por lotes desde disco, GPU, entrenamiento distribuido y costo, y en qué partes del código no tendrías que cambiar nada.

*Tu reflexión aquí.*

## Declaración de uso de IA

En este módulo **se permite el uso total de la IA** siempre que lo declares. Completa la tabla, incluyendo el desafío de la parte 6:

| | |
|---|---|
| **Herramienta y versión** | *por ejemplo: Gemini en Colab, noviembre de 2026* |
| **Para qué la usaste** | *script, guion de SLURM, depuración…* |
| **Prompts principales** | *copia los dos o tres más importantes* |
| **Qué verificaste tú** | *cómo comprobaste que el resultado era correcto* |
| **Qué corregiste** | *errores de la IA que encontraste y cómo los arreglaste* |

## Lista de control antes de entregar

- [ ] *Reiniciar y ejecutar todas* corre sin errores y las autoverificaciones muestran ✓.
- [ ] La prueba se usó **una sola vez** (parte 5); la parada temprana usó la validación.
- [ ] Completaste la pregunta 5, la ficha del modelo, la reflexión y la declaración de IA, y pegaste el script y el guion de SLURM.
- [ ] El archivo se llama `D3_Apellido_Nombre.ipynb` y lo subiste al aula virtual antes del jueves 19/11.

**Rúbrica (10 puntos):** reproducibilidad (versiones y semillas) 1.5 · flujo de PyTorch implementado 3 · validación honesta y comparación con cifras 2 · desafío con IA verificado y declarado 2 · reflexión sobre 100 veces más datos 1.5.